In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os
from dotenv import load_dotenv

import torch
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
client = QdrantClient("localhost", port=6333)

In [6]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
model_name = "intfloat/multilingual-e5-small"
model = SentenceTransformer(model_name, token=hf_token, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8378.25it/s]


In [7]:
if not client.collection_exists("legal_docs"):
    client.create_collection(
        collection_name="legal_docs",
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )

In [8]:
chunks = pd.read_json("../data/processed/chunks/trafik_kanunu.jsonl", lines=True)

In [9]:
chunk_ids = chunks["chunk_id"].tolist()
texts = chunks["text"].tolist()
metadatas = chunks["metadata"].tolist()

embeddings = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

In [10]:
points = [
    PointStruct(
        id=i,
        vector=emb.tolist(),
        payload={"chunk_id": chunk_id, "text": text_val, **metadata_val},
    )
    for i, (chunk_id, text_val, metadata_val, emb) in enumerate(
        zip(chunk_ids, texts, metadatas, embeddings)
    )
]

In [11]:
client.upsert(collection_name="legal_docs", points=points)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)